### Configuration

In [ ]:
import os 
from pathlib import Path
from tqdm import tqdm
from tqdm.auto import tqdm
import pandas as pd

import sys
import json
import numpy as np
import openai

In [ ]:
ROOT = Path(os.path.abspath('')).parent.parent
print(ROOT)
PROCESSED_FOLDER = ROOT / 'data/processed'
EMBEDDINGS_FOLDER = ROOT / 'data/embeddings'

EMBEDDINGS_FILE = EMBEDDINGS_FOLDER / 'open_ai_embeddings.npy'
CHUNKS_FILE = PROCESSED_FOLDER / 'all_chunks.jsonl'

# model
BATCH_SIZE = 100
MODEL_TO_USE = "text-embedding-3-small"
METADATA_FILE = EMBEDDINGS_FOLDER / 'open_ai_metadata.jsonl'


In [ ]:
client = openai.Client()

In [ ]:
chunks_df = pd.read_json(CHUNKS_FILE, lines=True)
chunks_df.head()

In [ ]:
texts_to_embed = chunks_df['page_content'].tolist()
print(f"Number of texts to embed: {len(texts_to_embed)}")

In [ ]:
metadata_to_save = chunks_df['metadata'].tolist()
print(f"Number of metadata to save: {len(metadata_to_save)}")

### Generate embeddings in batches

In [ ]:
all_embeddings = []

for i in tqdm(range(0, len(texts_to_embed), BATCH_SIZE), desc="Generating Embeddings"):
    batch_texts = texts_to_embed[i:i + BATCH_SIZE]
    
    try:
        response = client.embeddings.create(
            input=batch_texts,
            model=MODEL_TO_USE
        )
        # Extract the embedding vectors from the API response
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        
    except Exception as e:
        print(f"An error occurred while processing a batch: {e}", file=sys.stderr)
        break

embeddings_np = np.array(all_embeddings)

    

In [ ]:
embeddings_np.shape

In [ ]:
np.save(EMBEDDINGS_FILE, embeddings_np)

In [ ]:
pd.DataFrame(metadata_to_save).to_json(METADATA_FILE, orient='records', lines=True)
